In [1]:
import torch as pt


mols_test = pt.load('./data/mine/test_11499.pt')
print(len(mols_test))
mols_val = pt.load('./data/mine/val_11825.pt')
print(len(mols_val))
mols_all = pt.load('./data/mine/mols_all.pt')
print(len(mols_all))

/tmp/ipykernel_288914/3602205055.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mols_test = pt.load('./data/mine/test_11499.pt')


11499


/tmp/ipykernel_288914/3602205055.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mols_val = pt.load('./data/mine/val_11825.pt')


11825


/tmp/ipykernel_288914/3602205055.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mols_all = pt.load('./data/mine/mols_all.pt')


2253216


In [2]:
# 统计词频
import numpy as np


mols_train = mols_all[:232826]
count_list = np.zeros(1000)
for mol in mols_train:
    tmp_list = np.zeros(1000)
    for mz in mol.mz:
        tmp_list[int(mz)] = 1
    count_list += tmp_list

count_list += 1  
print(count_list.shape)

(1000,)


In [3]:
import numpy as np


# 生成负采样概率
pow_frequency = np.array(count_list) ** 0.75
neg_prob = pow_frequency / pow_frequency.sum()
print(neg_prob.shape)
# 生成下采样概率
mzs_freq = np.array(count_list)
mzs_freq = mzs_freq / np.sum(mzs_freq)
t = 1e-3
keep_prob = np.array([np.sqrt(t/f) + t/f for f in mzs_freq])

(1000,)


In [4]:
from torch.utils.data import DataLoader
from utils.data import SpecDataset, SpecDataset_finetune, collate_fun, collate_fun_emb, collate_fun_finetune


dataset_lib = SpecDataset(mols_all)
loader_lib = DataLoader(dataset_lib, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)
dataset_val = SpecDataset(mols_val)
loader_val = DataLoader(dataset_val, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)
dataset_test = SpecDataset(mols_test)
loader_test = DataLoader(dataset_test, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_emb)
dataset_finetune = SpecDataset_finetune((mols_val, mols_all))

In [10]:
dataset_train_mz = SpecDataset(dataset_lib, mapping=np.arange(232826))
loader_train_mz = DataLoader(dataset_train_mz, batch_size=32, shuffle=True,
                        num_workers=8, collate_fn=collate_fun(keep_prob, neg_prob))

dataset_train_inten = SpecDataset(dataset_lib, mapping=np.arange(232826))
loader_train_inten = DataLoader(dataset_train_inten, batch_size=1024, shuffle=True,
                        num_workers=8, collate_fn=collate_fun_emb)

In [6]:
import torch as pt
import torch.nn as nn
import torch.nn.functional as F


class IntenMlp(nn.Module):
    def __init__(self, in_dim:int=1000, out_dim:int=1000, hidden_dim:int=1000, hidden_num:int=1):
        super(IntenMlp, self).__init__()
        self.fc_in = nn.Linear(in_dim, hidden_dim)
        # 动态生成 hidden_num 个 (Linear, ReLU) 
        layers = []
        for _ in range(hidden_num):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
        self.fc_hidden = nn.Sequential(*layers) if layers else nn.Identity()
        self.fc_out = nn.Linear(hidden_dim, out_dim)
        self.mlp = nn.Sequential(self.fc_in, nn.ReLU(), self.fc_hidden, self.fc_out, nn.Sigmoid())

    def forward(self, x):
        mzs, intens, _ = x  # [batch, seq]
        intens_1000 = pt.zeros((intens.size(0), 1000), device=intens.device)  # [batch, 1000]
        intens_1000.scatter_(dim=1, index=mzs, src=intens)
        intens_1000 = self.mlp(intens_1000)  # [batch, 1000]
        # 再 gather 回来
        intens = intens_1000.gather(dim=1, index=mzs)  # [batch, seq]
        return intens


In [17]:
import torch.optim as optim
from utils.model import Spec2Emb, Linear_Scheduler
import torch.nn as nn
import torch.nn.functional as F


class Spec2Emb(nn.Module):
    def __init__(self, num_emb:int=1000, emb_dim:int=512, hidden_num:int=1):
        super(Spec2Emb, self).__init__()
        self.max_exp = 6
        self.emb_con = nn.Embedding(
            num_embeddings=num_emb,
            embedding_dim=emb_dim,
        )
        self.emb_cen = nn.Embedding(
            num_embeddings=num_emb,
            embedding_dim=emb_dim,
        )
        self.trip_loss = nn.TripletMarginLoss()
        self.burger = IntenMlp(hidden_num=hidden_num)
        
    def _compute_embedding(self, data, power, inten_mode:str='burger'):
        mzs, intens, masks = data  # [batch, seq]
        embs = self.emb_cen(mzs) # [batch, seq, emb_dim]
        # intens [batch, seq]
        embs = embs * masks.unsqueeze(-1)
        if inten_mode == 'burger':
            intens = self.burger(data)  # [batch, seq]
            embs = (embs * intens.unsqueeze(-1)).sum(dim=1) / intens.sum(dim=1, keepdim=True)
        elif inten_mode == 'pow':
            intens = pt.pow(intens, power)
            embs = (embs * intens.unsqueeze(-1)).sum(dim=1)
        else:
            raise ValueError('inten mode not exist')
        return embs

    def forward(self, data, mode:str='train_mz', power:float=0.5, inten_mode:str='pow'):
        if mode == 'train_mz': 
            # masks_con:
            mzs_con, masks_con, poss_cen, batch_idx, negs_cen, masks_neg = data
            embs_con = self.emb_con(mzs_con)        # [batch, seq, emb_dim]
            embs_pos = self.emb_cen(poss_cen)     # [B, emb_dim]
            embs_neg = self.emb_cen(negs_cen)      # [B, neg_num, emb_dim]
            embs_neg *= masks_neg.unsqueeze(-1)
            # for every cen word its context words
            embs_con = embs_con[batch_idx] * masks_con.unsqueeze(-1)
            embs_con = embs_con.sum(dim=1) / masks_con.sum(dim=1).unsqueeze(-1) # [B, emb_dim]
            # score
            pos_score = (embs_con * embs_pos).sum(dim=-1) # 点积
            pos_score = pt.clamp(pos_score, max=self.max_exp, min=-self.max_exp)
            pos_score = -F.logsigmoid(pos_score)
            neg_score = pt.bmm(embs_neg, embs_con.unsqueeze(-1)).squeeze(-1) #
            neg_score = pt.clamp(neg_score, max=self.max_exp, min=-self.max_exp)
            neg_score = -F.logsigmoid(-neg_score).sum(dim=-1)
            return (pos_score + neg_score).sum() 
        elif mode == 'train_inten':
            intens = self.burger(data)  # data: (mzs, intens)
            loss = F.huber_loss(intens, pt.sqrt(data[1]))
            return 1000*loss
        elif mode == 'emb': # emb模式下的masks只mask掉了padding             
            return self._compute_embedding(data, power, inten_mode)
        elif mode == 'finetune':
            data_mea, data_pre_hit, data_pre_nhit = data
            embs_mea = self._compute_embedding(data_mea, power)
            embs_pre_hit = self._compute_embedding(data_pre_hit, power)
            embs_pre_nhit = self._compute_embedding(data_pre_nhit, power)
            # batchsize, emb_dim
            embs_mea = F.normalize(embs_mea, p=2, dim=-1)
            embs_pre_hit = F.normalize(embs_pre_hit, p=2, dim=-1)
            embs_pre_nhit = F.normalize(embs_pre_nhit, p=2, dim=-1)
            # batchsize
            loss = self.trip_loss(embs_mea, embs_pre_hit, embs_pre_nhit)
            return loss
        else:
            raise ValueError('mode not exist')


gpu = 6
model = Spec2Emb(hidden_num=2).to(gpu)

epochs = 10
lr_inten = 0.001
optimizer_inten = optim.Adam(model.parameters(), lr=lr_inten)
lr_mz = 0.025
optimizer_mz = optim.SGD(model.parameters(), lr=lr_mz)
scheduler = Linear_Scheduler(optimizer_mz, epochs, start_lr=lr_mz, end_lr=2.5e-4)

In [18]:
from tqdm import tqdm
from utils.tools import build_idx, evaluate, save_model


def gen_embeddings(model:nn.Module, loader:DataLoader, gpu:int, power:float=0.5, inten_mode:str='burger'):
    model.eval()
    embs = []
    with pt.no_grad():
        for mzs_con, intens_con, masks in loader:
            data = [d.to(gpu) for d in (mzs_con, intens_con, masks)]
            emb = model(tuple(data), mode='emb', power=power, inten_mode=inten_mode).detach().cpu().numpy()
            embs.append(emb)
    pt.cuda.empty_cache()
    embs = np.concatenate(embs, axis=0)
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)
    return embs


num_batches = len(loader_train_mz)
model_name = 'inten_mlp_1000_1'
f = open(model_name + '.txt', 'w')
max_metrics = {'expand': [0, 0], 'insilico': [0, 0]}
for epoch in range(epochs):
    print(f'==================================Train_epoch{epoch+1}======================================')
    model.train()
    train_loss_inten = []
    for i, Data in enumerate(tqdm(loader_train_inten, unit='batch')):
        data = [d.to(gpu) for d in Data]
        optimizer_inten.zero_grad()
        loss = model(data, mode='train_inten')
        train_loss_inten.append(loss.item())
        loss.backward()
        optimizer_inten.step()
        if (i+1) %100 ==0:
            loss = np.mean(train_loss_inten)
            print(f'Inten Loss: {loss}')
            train_loss_inten = []
    
    train_loss_mz = []
    for i, Data in enumerate(tqdm(loader_train_mz, unit='batch')):
        data = [d.to(gpu) for d in Data]
        optimizer_mz.zero_grad()
        loss = model(data, mode='train_mz')
        train_loss_mz.append(loss.item())
        loss.backward()
        optimizer_mz.step()
        batch_progress = (i+1)/num_batches
        lr = scheduler.lr_lambda(epoch, batch_progress)
        for param_group in optimizer_mz.param_groups:
            param_group['lr'] = lr
        if (i+1) %1000 ==0:
            loss = np.mean(train_loss_mz)
            print(f'Total Loss: {loss}')
            train_loss_mz = []

    print(f'===================================Test_epoch{epoch+1}======================================')
    f.write('\nTest_epoch%d\n' % (epoch+1))
    embeddings_lib = gen_embeddings(model, loader_lib, gpu)
    embeddings_test = gen_embeddings(model, loader_test, gpu)
    I_expand, _ = build_idx(embeddings_lib, embeddings_test, gpu)
    top1_expand, top10_expand = evaluate(mols_test, I_expand, mols_all, f, 'Expanded')
    if top1_expand > max_metrics['expand'][0] and top10_expand > max_metrics['expand'][1]:
        max_metrics['expand'] = [top1_expand, top10_expand]
        save_model(model, model_name, epoch)
    I_insilico, _ = build_idx(embeddings_lib[:2146690], embeddings_test, gpu)
    top1_insilico, top10_insilico = evaluate(mols_test, I_insilico, mols_all, f, 'In-silico')
    if top1_insilico > max_metrics['insilico'][0] and top10_insilico > max_metrics['insilico'][1]:
        max_metrics['insilico'] = [top1_insilico, top10_insilico]
        save_model(model, model_name, epoch)
    print(f'================================================================================================')
f.close()

==================================Train_epoch1======================================


  0%|          | 0/228 [00:00<?, ?batch/s]

 47%|████▋     | 108/228 [00:05<00:02, 43.96batch/s]

Inten Loss: 7.3602307772636415


 89%|████████▉ | 204/228 [00:07<00:00, 51.84batch/s]

...

 96%|█████████▋| 7010/7276 [02:07<00:04, 63.31batch/s]

Total Loss: 4316.596219238281


100%|██████████| 7276/7276 [02:13<00:00, 54.37batch/s]

===================================Test_epoch10======================================


Searching time:  0:00:01.544378
Expanded library
Top1 hit rate: 24.05%
Top10 hit rate: 57.42%
Searching time:  0:00:01.475509
In-silico library
Top1 hit rate: 24.29%
Top10 hit rate: 57.97%
Model already exists!


In [9]:
pt.cuda.empty_cache()